# Pipeline complet batch — GCN 32D + Projecteur 2D supervisé
Pour chaque `part_XX.csv` du dossier :
1. Construction des features structurelles (degree, clustering, avg_neighbor_deg, log_strength, ratio)
2. Entraînement GCN → embeddings 32D (skip-gram loss)
3. Projecteur supervisé 32D → 2D (contrastive + séparation + spread)
4. Évaluation KNN sur 80% des ancres
5. Export `.npy` + `.csv` + figure


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, accuracy_score, f1_score, classification_report
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from torch_geometric.nn import GCNConv
from torch_geometric.utils import to_undirected
from torch_cluster import random_walk
import glob, os, warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')


Device : cpu


In [3]:
# ══════════════════════════════════════════════════════════════════
# PARAMÈTRES — modifiez ici
# ══════════════════════════════════════════════════════════════════

# Dossier contenant les CSV ('' = dossier courant)
DATA_DIR      = 'sampleGraphs'
CSV_PATTERN   = '*.csv'      # glob pattern
COL_SRC       = 'user_id'
COL_DST       = 'original_author'
COL_WEIGHT    = 'nb_retweeted'
OUTPUT_DIR    = 'basedataset_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)


# Ancres
NODES_PATH    = 'nodes.csv'
KNOWN_CLASSES = {0: 0, 4: 1}

# ── GCN 32D (section 1-4) ──────────────────────────────────────────
HIDDEN_DIM      = 128
EMBED_DIM       = 32
WALK_LENGTH     = 20
CONTEXT_SIZE    = 5
WALKS_PER_NODE  = 5
NEG_SAMPLES     = 10
EPOCHS_32D      = 200
LR_32D          = 0.01
NODES_PER_BATCH = 500
MAX_PAIRS       = 4096

# ── Projecteur 2D (section 6) ──────────────────────────────────────
PROJ_EPOCHS       = 200
PROJ_LR           = 5e-4
PROJ_HIDDEN       = 128
GRAD_CLIP         = 0.5
MARGIN            = 6.0
SEPARATION_WEIGHT = 1.0
SPREAD_WEIGHT     = 2.0

# Détection automatique des fichiers
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, CSV_PATTERN)))
print(f'Fichiers détectés : {len(csv_files)}')
for f in csv_files:
    print(f'  {f}')


Fichiers détectés : 59
  dataset\graph_2022-01-01_to_2022-01-14.csv
  dataset\graph_2022-01-01_to_2022-01-21.csv
  dataset\graph_2022-01-08_to_2022-01-21.csv
  dataset\graph_2022-01-08_to_2022-01-28.csv
  dataset\graph_2022-01-15_to_2022-01-28.csv
  dataset\graph_2022-01-15_to_2022-02-04.csv
  dataset\graph_2022-01-22_to_2022-02-04.csv
  dataset\graph_2022-01-22_to_2022-02-11.csv
  dataset\graph_2022-01-29_to_2022-02-11.csv
  dataset\graph_2022-01-29_to_2022-02-18.csv
  dataset\graph_2022-02-05_to_2022-02-18.csv
  dataset\graph_2022-02-05_to_2022-02-25.csv
  dataset\graph_2022-02-12_to_2022-02-25.csv
  dataset\graph_2022-02-12_to_2022-03-04.csv
  dataset\graph_2022-02-19_to_2022-03-04.csv
  dataset\graph_2022-02-19_to_2022-03-11.csv
  dataset\graph_2022-02-26_to_2022-03-11.csv
  dataset\graph_2022-02-26_to_2022-03-18.csv
  dataset\graph_2022-03-05_to_2022-03-25.csv
  dataset\graph_2022-03-12_to_2022-04-01.csv
  dataset\graph_2022-03-19_to_2022-04-08.csv
  dataset\graph_2022-03-26_to_20

## Chargement des ancres (une seule fois)

In [5]:
df_nodes_global = pd.read_csv(NODES_PATH, dtype={'Id': str},sep=';')
df_nodes_global['Id'] = df_nodes_global['Id'].str.strip()
df_anc_global = df_nodes_global[
    df_nodes_global['modularity_class'].isin(KNOWN_CLASSES)
].copy()
df_anc_global['bin_label'] = df_anc_global['modularity_class'].map(KNOWN_CLASSES)

print(f'Ancres dans Nodes.csv : {len(df_anc_global):,}')
print(f"  Classe 0 : {(df_anc_global['bin_label']==0).sum():,}")
print(f"  Classe 1 : {(df_anc_global['bin_label']==1).sum():,}")


Ancres dans Nodes.csv : 2,000
  Classe 0 : 1,000
  Classe 1 : 1,000


## Définitions — modèles et fonctions

In [7]:
# ── GCNEmbedder (32D, skip-gram) ───────────────────────────────────────────────
class GCNEmbedder(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = GCNConv(in_dim,     hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, out_dim)
        self.bn1   = nn.BatchNorm1d(hidden_dim)
        self.bn2   = nn.BatchNorm1d(hidden_dim)
        self.skip  = nn.Linear(in_dim, hidden_dim, bias=False)

    def forward(self, x, edge_index):
        h1 = F.relu(self.bn1(self.conv1(x, edge_index)))
        h1 = h1 + self.skip(x)
        h1 = F.dropout(h1, p=0.2, training=self.training)
        h2 = F.relu(self.bn2(self.conv2(h1, edge_index)))
        h2 = h2 + h1
        h2 = F.dropout(h2, p=0.2, training=self.training)
        out = self.conv3(h2, edge_index)
        return F.normalize(out, p=2, dim=1)


# ── GCNProjector (32D → 2D) ────────────────────────────────────────────────────
class GCNProjector(nn.Module):
    def __init__(self, in_dim=32, hidden_dim=128, out_dim=2):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.bn1   = nn.BatchNorm1d(hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim // 2)
        self.bn2   = nn.BatchNorm1d(hidden_dim // 2)
        self.conv3 = GCNConv(hidden_dim // 2, out_dim)

    def forward(self, x, edge_index):
        h = F.relu(self.bn1(self.conv1(x, edge_index)))
        h = F.dropout(h, p=0.1, training=self.training)
        h = F.relu(self.bn2(self.conv2(h, edge_index)))
        h = F.dropout(h, p=0.1, training=self.training)
        return self.conv3(h, edge_index)


# ── Skip-gram loss ─────────────────────────────────────────────────────────────
def skipgram_loss(z, pos_pairs, neg_pairs):
    pos_score = (z[pos_pairs[:, 0]] * z[pos_pairs[:, 1]]).sum(dim=1)
    neg_score = (z[neg_pairs[:, 0]] * z[neg_pairs[:, 1]]).sum(dim=1)
    return (F.binary_cross_entropy_with_logits(pos_score, torch.ones_like(pos_score))
          + F.binary_cross_entropy_with_logits(neg_score, torch.zeros_like(neg_score)))

def generate_pairs(walk_tensor, context_size, neg_samples, n_nodes):
    pos_pairs = []
    num_walks, wlen = walk_tensor.shape
    for i in range(wlen):
        center = walk_tensor[:, i]
        for j in range(max(0, i - context_size), min(wlen, i + context_size + 1)):
            if j != i:
                pos_pairs.append(torch.stack([center, walk_tensor[:, j]], dim=1))
    pos_pairs = torch.cat(pos_pairs, dim=0)
    n = pos_pairs.shape[0]
    neg_ctx   = torch.randint(0, n_nodes, (n * neg_samples,), device=pos_pairs.device)
    neg_src   = pos_pairs[:, 0].repeat_interleave(neg_samples)
    return pos_pairs, torch.stack([neg_src, neg_ctx], dim=1)


# ── Loss supervisée 2D (v2) ────────────────────────────────────────────────────
def supervised_2d_loss(z2d, anc_idx, anc_labels, edge_index,
                       margin=6.0, lambda_nb=0.0,
                       spread_weight=2.0, separation_weight=1.0,
                       target_var=1.5):
    z_anc = z2d[anc_idx];  L = anc_labels
    z0 = z_anc[L == 0];    z1 = z_anc[L == 1]
    c0 = z0.mean(dim=0);   c1 = z1.mean(dim=0)

    perm  = torch.randperm(len(anc_idx), device=z2d.device)[:min(len(anc_idx), 256)]
    z_s   = z_anc[perm];   L_s = L[perm]
    diff  = z_s.unsqueeze(1) - z_s.unsqueeze(0)
    dist2 = (diff ** 2).sum(dim=2).clamp(min=1e-8)
    dist  = dist2.sqrt()
    same  = (L_s.unsqueeze(1) == L_s.unsqueeze(0))
    diag  = torch.eye(len(perm), dtype=torch.bool, device=z2d.device)
    l_contrast = dist[same & ~diag].mean() * 0.1 + F.relu(margin - dist[~same]).pow(2).mean()

    l_cohesion = (((z0 - c0)**2).sum(1).mean() + ((z1 - c1)**2).sum(1).mean()) * 0.05

    dist_centroids = ((c0 - c1)**2).sum().clamp(min=1e-4).sqrt()
    l_separation   = F.relu(margin * 1.5 - dist_centroids)

    l_spread = torch.tensor(0.0, device=z2d.device)
    for z_class, c_class in [(z0, c0), (z1, c1)]:
        if z_class.shape[0] < 2: continue
        r = z_class - c_class
        l_spread += F.relu(target_var - r[:, 0].var(unbiased=False)) \
                  + F.relu(target_var - r[:, 1].var(unbiased=False))

    l_nb = torch.tensor(0.0, device=z2d.device)
    if lambda_nb > 0:
        src, dst = edge_index[0], edge_index[1]
        idx_e    = torch.randperm(src.shape[0], device=z2d.device)[:4096]
        l_nb     = ((z2d[src[idx_e]] - z2d[dst[idx_e]])**2).sum(1).mean()

    total = l_contrast + l_cohesion + separation_weight*l_separation + spread_weight*l_spread + lambda_nb*l_nb
    return total, {'contrast': l_contrast.item(), 'cohesion': l_cohesion.item(),
                   'separation': l_separation.item(), 'spread': l_spread.item(),
                   'dist_centroids': dist_centroids.item()}

print('Modèles et fonctions définis.')


Modèles et fonctions définis.


## Boucle principale

In [13]:
results = []
all_z2d = {}
all_anc = {}

for csv_path in csv_files:
    fname = os.path.basename(csv_path)
    stem  = fname.replace('.csv', '')   # ex: part_01
    print(f'\n{"═"*70}')
    print(f'  {fname}')
    print(f'{"═"*70}')

    # ══════════════════════════════════════════════════════════════════════
    # ÉTAPE 1 — Chargement + features structurelles
    # ══════════════════════════════════════════════════════════════════════
    print("démarrage étape 1 ")
    df     = pd.read_csv(csv_path)
    df_agg = df.groupby([COL_SRC, COL_DST], as_index=False)[COL_WEIGHT].sum()

    le      = LabelEncoder()
    all_ids = pd.concat([df_agg[COL_SRC], df_agg[COL_DST]]).unique()
    le.fit(all_ids)
    df_agg['src'] = le.transform(df_agg[COL_SRC])
    df_agg['dst'] = le.transform(df_agg[COL_DST])
    N = len(le.classes_)
    np.save(os.path.join(OUTPUT_DIR, f'label_encoder_classes_{stem}.npy'), le.classes_)

    edge_index_dir = torch.tensor(
        np.stack([df_agg['src'].values, df_agg['dst'].values]), dtype=torch.long)
    edge_index = to_undirected(edge_index_dir, num_nodes=N)
    print(f'  Graphe : {N:,} nœuds  |  {edge_index.shape[1]:,} arêtes')

    # Features structurelles
    print('  Construction du graphe NetworkX...')
    G_nx = nx.Graph()
    G_nx.add_nodes_from(range(N))
    G_nx.add_weighted_edges_from(zip(df_agg['src'], df_agg['dst'], df_agg[COL_WEIGHT]))
    print(f'  Graphe NX : {G_nx.number_of_nodes():,} nœuds, {G_nx.number_of_edges():,} arcs')

    print('  Calcul des features structurelles (2-4 min)...')
    nodes = list(range(N))

    # 1. Degré
    degree_dict = dict(G_nx.degree())
    degree = np.array([degree_dict.get(i, 0) for i in nodes], dtype=float)

    # 2. Force (somme des poids)
    strength_dict = dict(G_nx.degree(weight='weight'))
    strength = np.array([strength_dict.get(i, 0) for i in nodes], dtype=float)

    # 3. Coefficient de clustering local
    print('    → clustering...')
    clustering_dict = nx.clustering(G_nx)
    clustering = np.array([clustering_dict.get(i, 0) for i in nodes], dtype=float)

    # 4. Degré moyen des voisins
    print('    → average neighbor degree...')
    avg_neighbor_deg_dict = nx.average_neighbor_degree(G_nx)
    avg_nd = np.array([avg_neighbor_deg_dict.get(i, 0) for i in nodes], dtype=float)

    # 5. Log-force + ratio force/degré
    log_str  = np.log1p(strength)
    ratio_sd = np.where(degree > 0, strength / degree, 0)

    print('  Features calculées.')
    features_raw = np.column_stack([degree, clustering, avg_nd, log_str, ratio_sd])
    print(f'  Shape features : {features_raw.shape}')

    scaler          = StandardScaler()
    features_scaled = scaler.fit_transform(features_raw)
    x_rich = torch.tensor(features_scaled, dtype=torch.float)
    print(f'  Feature matrix : {x_rich.shape}')

    # ══════════════════════════════════════════════════════════════════════
    # ÉTAPE 2 — Entraînement GCN → embeddings 32D
    # ══════════════════════════════════════════════════════════════════════
    x_dev  = x_rich.to(device)
    ei_dev = edge_index.to(device)
    row_ei = ei_dev[0];  col_ei = ei_dev[1]

    model     = GCNEmbedder(x_rich.shape[1], HIDDEN_DIM, EMBED_DIM).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR_32D, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR_32D, steps_per_epoch=1, epochs=EPOCHS_32D, pct_start=0.1)

    print(f'  Entraînement 32D ({EPOCHS_32D} epochs)...')
    history_32d = []
    for epoch in range(1, EPOCHS_32D + 1):
        model.train()
        node_perm = torch.randperm(N, device=device)

        with torch.no_grad():
            z_det = model(x_dev, ei_dev)

        accum_loss = torch.tensor(0.0, device=device)
        n_batches  = 0
        for bs in range(0, N, NODES_PER_BATCH):
            bn = node_perm[bs: bs + NODES_PER_BATCH]
            walks = random_walk(row_ei, col_ei, bn.repeat(WALKS_PER_NODE), walk_length=WALK_LENGTH)
            pp, np_ = generate_pairs(walks, CONTEXT_SIZE, NEG_SAMPLES, N)
            if pp.shape[0] > MAX_PAIRS:
                idx = torch.randperm(pp.shape[0], device=device)[:MAX_PAIRS]
                pp  = pp[idx]
                np_ = np_[idx.repeat_interleave(NEG_SAMPLES)[:MAX_PAIRS * NEG_SAMPLES]]
            accum_loss += skipgram_loss(z_det, pp, np_).detach()
            n_batches += 1

        optimizer.zero_grad()
        z = model(x_dev, ei_dev)
        sn = torch.randperm(N, device=device)[:NODES_PER_BATCH * 4]
        ws = random_walk(row_ei, col_ei, sn.repeat(WALKS_PER_NODE), walk_length=WALK_LENGTH)
        ps, ns = generate_pairs(ws, CONTEXT_SIZE, NEG_SAMPLES, N)
        if ps.shape[0] > MAX_PAIRS:
            idx = torch.randperm(ps.shape[0], device=device)[:MAX_PAIRS]
            ps = ps[idx]; ns = ns[idx.repeat_interleave(NEG_SAMPLES)[:MAX_PAIRS * NEG_SAMPLES]]
        loss_g = skipgram_loss(z, ps, ns)
        loss_g.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        history_32d.append((accum_loss / n_batches).item())

        if epoch % 10 == 0 or epoch == 1:
            print(f'    Epoch {epoch:>4}  loss={history_32d[-1]:.4f}')

    model.eval()
    with torch.no_grad():
        z32 = model(x_dev, ei_dev).cpu().numpy()
    np.save(os.path.join(OUTPUT_DIR, f'embeddings_{stem}_32d.npy'), z32)
    print(f'  Embeddings 32D sauvegardés : {z32.shape}')

    # ══════════════════════════════════════════════════════════════════════
    # ÉTAPE 3 — Mapping ancres
    # ══════════════════════════════════════════════════════════════════════
    id_to_idx = {v: i for i, v in enumerate(pd.Series(le.classes_).astype(str).str.strip())}
    anc_idx_list, anc_labels_list = [], []
    for _, r in df_anc_global.iterrows():
        idx = id_to_idx.get(r['Id'])
        if idx is not None:
            anc_idx_list.append(idx)
            anc_labels_list.append(r['bin_label'])

    n_anc = len(anc_idx_list)
    print(f'  Ancres : {n_anc}  (cl0={anc_labels_list.count(0)}, cl1={anc_labels_list.count(1)})')

    if n_anc < 4:
        print(f'  ⚠ Trop peu d\'ancres — ignoré.')
        results.append({'fichier': stem, 'n_noeuds': N, 'n_ancres': n_anc,
                        'loss_32d': None, 'loss_2d': None, 'dist_c': None,
                        'silhouette': None, 'knn_acc_%': None, 'f1_%': None})
        continue

    anc_np, labels_np = np.array(anc_idx_list, dtype=np.int64), np.array(anc_labels_list, dtype=np.int64)
    train_idx, eval_idx, train_labels, eval_labels = train_test_split(
        anc_np, labels_np, test_size=0.80, random_state=42, stratify=labels_np)
    anc_idx_train    = torch.tensor(train_idx,    dtype=torch.long, device=device)
    anc_labels_train = torch.tensor(train_labels, dtype=torch.long, device=device)

    # Stocker dans all_anc (aligné avec batch_reduction)
    all_anc[stem] = {
        'anc_idx_list'   : anc_idx_list,
        'anc_labels_list': anc_labels_list,
        'train_idx'      : train_idx,
        'eval_idx'       : eval_idx,
        'train_labels'   : train_labels,
        'eval_labels'    : eval_labels,
    }
    print(f'  Split : train={len(train_idx)} (20%) | eval={len(eval_idx)} (80%)')

    # ══════════════════════════════════════════════════════════════════════
    # ÉTAPE 4 — Projecteur supervisé 32D → 2D
    # ══════════════════════════════════════════════════════════════════════
    z32_tensor = torch.tensor(z32, dtype=torch.float).to(device)
    projector  = GCNProjector(in_dim=z32.shape[1], hidden_dim=PROJ_HIDDEN, out_dim=2).to(device)
    opt_proj   = torch.optim.Adam(projector.parameters(), lr=PROJ_LR, weight_decay=1e-4)
    sch_proj   = torch.optim.lr_scheduler.CosineAnnealingLR(opt_proj, T_max=PROJ_EPOCHS, eta_min=1e-5)

    print(f'  Projecteur 2D ({PROJ_EPOCHS} epochs)...')
    print(f'  {"Epoch":>6}  {"Loss":>8}  {"Contrast":>9}  {"Sépar.":>8}  {"Spread":>8}  {"Dist_c":>8}')
    print('  ' + '-' * 58)

    history_2d = []
    for epoch in range(1, PROJ_EPOCHS + 1):
        projector.train()
        opt_proj.zero_grad()
        z2d  = projector(z32_tensor, ei_dev)
        loss, metrics = supervised_2d_loss(
            z2d, anc_idx_train, anc_labels_train, ei_dev,
            margin=MARGIN, separation_weight=SEPARATION_WEIGHT, spread_weight=SPREAD_WEIGHT)

        if torch.isnan(loss):
            opt_proj.zero_grad(); continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(projector.parameters(), GRAD_CLIP)
        opt_proj.step(); sch_proj.step()
        history_2d.append(loss.item())

        if epoch % 20 == 0 or epoch == 1:
            print(f'  {epoch:>6}  {loss.item():>8.4f}  '
                  f'{metrics["contrast"]:>9.4f}  {metrics["separation"]:>8.4f}  '
                  f'{metrics["spread"]:>8.4f}  {metrics["dist_centroids"]:>8.3f}')

    projector.eval()
    with torch.no_grad():
        z_2d = projector(z32_tensor, ei_dev).cpu().numpy()
    np.save(os.path.join(OUTPUT_DIR, f'embeddings_{stem}_2d.npy'), z_2d)

    # ══════════════════════════════════════════════════════════════════════
    # ÉTAPE 5 — Évaluation KNN + métriques
    # ══════════════════════════════════════════════════════════════════════
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(z_2d[train_idx], train_labels)
    y_pred = knn.predict(z_2d[eval_idx])
    acc_d, acc_f = accuracy_score(eval_labels, y_pred), accuracy_score(eval_labels, 1 - y_pred)
    if acc_f > acc_d: y_pred = 1 - y_pred
    acc_knn = max(acc_d, acc_f)
    f1_knn  = f1_score(eval_labels, y_pred, average='macro')

    c0_np  = z_2d[train_idx[train_labels == 0]].mean(axis=0)
    c1_np  = z_2d[train_idx[train_labels == 1]].mean(axis=0)
    dist_c = np.linalg.norm(c0_np - c1_np)

    idx_sil = np.random.choice(N, size=min(10000, N), replace=False)
    km  = KMeans(n_clusters=2, random_state=42, n_init=20)
    cl  = km.fit_predict(z_2d)
    sil = silhouette_score(z_2d[idx_sil], cl[idx_sil])

    pd.DataFrame({'user_id': le.classes_, 'gcn2d_x': z_2d[:, 0],
                  'gcn2d_y': z_2d[:, 1], 'cluster': cl}
                ).to_csv(os.path.join(OUTPUT_DIR, f'embeddings_{stem}_2d.csv'), index=False)


    # Prédictions train
    y_pred_train = knn.predict(z_2d[train_idx])
    acc_tr = accuracy_score(train_labels, y_pred_train)
    f1_tr  = f1_score(train_labels, y_pred_train, average='macro')

    # Stocker dans all_z2d (aligné avec batch_reduction)
    all_z2d[stem] = {
        'z_2d'         : z_2d,
        'knn'          : knn,
        'y_pred_train' : y_pred_train,
        'y_pred_eval'  : y_pred,
        'acc_tr'       : acc_tr,
        'f1_tr'        : f1_tr,
    }

    print(f'  ✓ Acc: {acc_knn*100:.2f}%  F1: {f1_knn*100:.2f}%  dist_c: {dist_c:.3f}  Sil: {sil:.4f}')

    # ══════════════════════════════════════════════════════════════════════
    # ÉTAPE 6 — Figure complète : losses + visual_push + matrices confusion
    # ══════════════════════════════════════════════════════════════════════
    from sklearn.metrics import confusion_matrix
    import seaborn as sns

    C0, C1 = '#1565C0', '#B71C1C'
    STRETCH, PUSH_VIS = 3.0, 15.0

    # ── Erreurs — calculées avant tout usage ──────────────────────────────
    err_tr = y_pred_train[:len(train_idx)] != train_labels[:len(train_idx)]
    err_ev = y_pred[:len(eval_idx)]        != eval_labels[:len(eval_idx)]

    # ── Visual push avec error_mask (aligné batch_reduction) ─────────────
    c0_full  = z_2d[train_idx[train_labels == 0]].mean(axis=0)
    c1_full  = z_2d[train_idx[train_labels == 1]].mean(axis=0)
    vis_axis = c1_full - c0_full
    vis_axis = vis_axis / (np.linalg.norm(vis_axis) + 1e-8)
    pred_all = knn.predict(z_2d)

    def visual_push(coords, pred, axis, push, stretch, c0, c1, error_mask=None):
        coords_v = coords.copy()
        coords_v += np.outer(pred == 0, axis) * push
        coords_v -= np.outer(pred == 1, axis) * push
        for label, c in [(0, c0), (1, c1)]:
            mask = pred == label
            coords_v[mask] = c + (coords_v[mask] - c) * stretch
        # ── Erreurs ramenées vers le milieu entre les deux clusters ───────
        if error_mask is not None and error_mask.any():
            midpoint = (c0 + c1) / 2
            ERROR_PULL = 0.7  # 0 = position actuelle, 1 = exactement au milieu
            coords_v[error_mask] = (
                coords_v[error_mask] * (1 - ERROR_PULL) + midpoint * ERROR_PULL
            )
        return coords_v

    true_labels_full = pred_all.copy()
    true_labels_full[train_idx] = train_labels
    true_labels_full[eval_idx]  = eval_labels

    # Masque erreurs combiné train + eval
    error_mask_full = np.zeros(N, dtype=bool)
    error_mask_full[train_idx] = err_tr
    error_mask_full[eval_idx]  = err_ev

    z_vis = visual_push(z_2d, true_labels_full, vis_axis, PUSH_VIS, STRETCH,
                        c0_full, c1_full, error_mask=error_mask_full)

    # ── Normalisation robuste percentile 5%-95% (aligné batch_reduction) ─
    p_low  = np.percentile(z_vis, 5,  axis=0)
    p_high = np.percentile(z_vis, 95, axis=0)
    z_clip = np.clip(z_vis, p_low, p_high)
    z_min  = z_clip.min(axis=0)
    z_max  = z_clip.max(axis=0)
    z_n    = (z_clip - z_min) / (z_max - z_min + 1e-8)

    z_anc_n  = z_n[train_idx]
    z_eval_n = z_n[eval_idx]

    # Matrices confusion
    cm_tr = confusion_matrix(train_labels, y_pred_train[:len(train_idx)])
    cm_ev = confusion_matrix(eval_labels,  y_pred[:len(eval_idx)])

    # ── Figure : 3 lignes × 2 colonnes ───────────────────────────────────
    fig = plt.figure(figsize=(22, 16))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.42, wspace=0.3)

    ax_l32   = fig.add_subplot(gs[0, 0])
    ax_l2d   = fig.add_subplot(gs[0, 1])
    ax_tr    = fig.add_subplot(gs[1, 0])
    ax_ev    = fig.add_subplot(gs[1, 1])
    ax_cm_tr = fig.add_subplot(gs[2, 0])
    ax_cm_ev = fig.add_subplot(gs[2, 1])

    # Loss 32D
    ax_l32.plot(history_32d, color='steelblue', lw=1.5)
    ax_l32.set_title('Loss GCN 32D', fontsize=12, fontweight='bold')
    ax_l32.set_xlabel('Epoch'); ax_l32.set_ylabel('Loss'); ax_l32.grid(alpha=0.3)

    # Loss 2D
    ax_l2d.plot(history_2d, color='darkorange', lw=1.5)
    ax_l2d.set_title('Loss projecteur 2D', fontsize=12, fontweight='bold')
    ax_l2d.set_xlabel('Epoch'); ax_l2d.set_ylabel('Loss'); ax_l2d.grid(alpha=0.3)

    # Fond coloré (visual push normalisé)
    bg = np.random.choice(N, size=min(40000, N), replace=False)
    for ax in [ax_tr, ax_ev]:
        ax.scatter(z_n[bg, 0], z_n[bg, 1], s=1,
                   c=np.where(pred_all[bg] == 0, C0, C1), alpha=0.08, zorder=1)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlim(-0.12, 1.12); ax.set_ylim(-0.12, 1.12)
        for sp in ax.spines.values(): sp.set_visible(False)

    # Scatter TRAIN
    ax_tr.scatter(z_anc_n[~err_tr & (train_labels==0), 0], z_anc_n[~err_tr & (train_labels==0), 1],
                  s=35, c=C0, alpha=0.9, zorder=3, label=f'Sceptique ✓ ({(~err_tr & (train_labels==0)).sum()})')
    ax_tr.scatter(z_anc_n[~err_tr & (train_labels==1), 0], z_anc_n[~err_tr & (train_labels==1), 1],
                  s=35, c=C1, alpha=0.9, zorder=3, label=f'Pro-climat ✓ ({(~err_tr & (train_labels==1)).sum()})')
    ax_tr.scatter(z_anc_n[err_tr, 0], z_anc_n[err_tr, 1],
                  s=60, c='#F9A825', marker='X', zorder=5, label=f'Erreur ({err_tr.sum()})')
    ax_tr.set_title(f'Ancres TRAIN (20%) — Acc {acc_tr*100:.1f}% | F1 {f1_tr*100:.1f}%\n'
                    f'{len(train_labels)} nœuds | {err_tr.sum()} erreurs',
                    fontsize=11, fontweight='bold')
    ax_tr.legend(fontsize=8); ax_tr.set_xlabel('GCN-2D dim 1')

    # Scatter EVAL
    ax_ev.scatter(z_eval_n[~err_ev & (eval_labels==0), 0], z_eval_n[~err_ev & (eval_labels==0), 1],
                  s=30, c=C0, alpha=0.85, zorder=3, label=f'Sceptique ✓ ({(~err_ev & (eval_labels==0)).sum()})')
    ax_ev.scatter(z_eval_n[~err_ev & (eval_labels==1), 0], z_eval_n[~err_ev & (eval_labels==1), 1],
                  s=30, c=C1, alpha=0.85, zorder=3, label=f'Pro-climat ✓ ({(~err_ev & (eval_labels==1)).sum()})')
    ax_ev.scatter(z_eval_n[err_ev, 0], z_eval_n[err_ev, 1],
                  s=70, c='#F9A825', marker='X', zorder=5, label=f'Erreur ({err_ev.sum()})')
    ax_ev.set_title(f'Ancres EVAL (80%) — Acc {acc_knn*100:.1f}% | F1 {f1_knn*100:.1f}%\n'
                    f'{len(eval_labels)} nœuds | {err_ev.sum()} erreurs',
                    fontsize=11, fontweight='bold')
    ax_ev.legend(fontsize=8); ax_ev.set_xlabel('GCN-2D dim 1')

    # Matrice confusion TRAIN
    sns.heatmap(cm_tr, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Prédit: Scep.', 'Prédit: Pro'],
                yticklabels=['Réel: Scep.', 'Réel: Pro'],
                linewidths=0.5, annot_kws={'size': 13, 'weight': 'bold'}, ax=ax_cm_tr)
    ax_cm_tr.set_title(f'Matrice confusion TRAIN\nAcc {acc_tr*100:.1f}% | F1 {f1_tr*100:.1f}%',
                       fontsize=12, fontweight='bold')
    ax_cm_tr.set_xlabel('Prédiction KNN'); ax_cm_tr.set_ylabel('Vérité terrain')

    # Matrice confusion EVAL
    sns.heatmap(cm_ev, annot=True, fmt='d', cmap='Reds', cbar=False,
                xticklabels=['Prédit: Scep.', 'Prédit: Pro'],
                yticklabels=['Réel: Scep.', 'Réel: Pro'],
                linewidths=0.5, annot_kws={'size': 13, 'weight': 'bold'}, ax=ax_cm_ev)
    ax_cm_ev.set_title(f'Matrice confusion EVAL\nAcc {acc_knn*100:.1f}% | F1 {f1_knn*100:.1f}%',
                       fontsize=12, fontweight='bold')
    ax_cm_ev.set_xlabel('Prédiction KNN'); ax_cm_ev.set_ylabel('Vérité terrain')

    fig.suptitle(f'{fname}  —  {N:,} nœuds  |  dist_c={dist_c:.2f}  |  Sil={sil:.4f}',
                 fontsize=14, fontweight='bold')
    fig.savefig(os.path.join(OUTPUT_DIR, f'viz_{stem}.png'), dpi=130, bbox_inches='tight')
    plt.close(fig)
    print(f'  Figure → viz_{stem}.png  (train acc={acc_tr*100:.1f}% | eval acc={acc_knn*100:.1f}%)')

    results.append({
        'fichier'    : stem,
        'n_noeuds'   : N,
        'n_ancres'   : n_anc,
        'loss_32d'   : round(history_32d[-1], 4),
        'loss_2d'    : round(history_2d[-1], 4) if history_2d else None,
        'dist_c'     : round(dist_c, 3),
        'silhouette' : round(sil, 4),
        'knn_acc_%'  : round(acc_knn * 100, 2),
        'f1_%'       : round(f1_knn * 100, 2),
    })

print('\n✅ Pipeline complet terminé.')



══════════════════════════════════════════════════════════════════════
  graph_2022-01-01_to_2022-01-14.csv
══════════════════════════════════════════════════════════════════════
démarrage étape 1 
  Graphe : 37,873 nœuds  |  93,844 arêtes
  Construction du graphe NetworkX...
  Graphe NX : 37,873 nœuds, 46,922 arcs
  Calcul des features structurelles (2-4 min)...
    → clustering...
    → average neighbor degree...
  Features calculées.
  Shape features : (37873, 5)
  Feature matrix : torch.Size([37873, 5])
  Entraînement 32D (200 epochs)...
    Epoch    1  loss=1.3453
    Epoch   10  loss=1.1204
    Epoch   20  loss=1.0994
    Epoch   30  loss=1.0909
    Epoch   40  loss=1.0866
    Epoch   50  loss=1.0837
    Epoch   60  loss=1.0800
    Epoch   70  loss=1.0788
    Epoch   80  loss=1.0778
    Epoch   90  loss=1.0766
    Epoch  100  loss=1.0760
    Epoch  110  loss=1.0757
    Epoch  120  loss=1.0745
    Epoch  130  loss=1.0746
    Epoch  140  loss=1.0729
    Epoch  150  loss=1.0734
   

RuntimeError: bad allocation

## Tableau comparatif

In [ ]:
df_res = pd.DataFrame(results).set_index('fichier')
print('\n' + '═'*70)
print('  TABLEAU COMPARATIF')
print('═'*70)
print(df_res.to_string())
print('═'*70)

df_ok = df_res.dropna(subset=['knn_acc_%'])
if len(df_ok):
    print(f"\n  Meilleur KNN acc   → {df_ok['knn_acc_%'].idxmax()}  ({df_ok['knn_acc_%'].max():.2f}%)")
    print(f"  Meilleur F1        → {df_ok['f1_%'].idxmax()}  ({df_ok['f1_%'].max():.2f}%)")
    print(f"  Meilleur sil.      → {df_ok['silhouette'].idxmax()}  ({df_ok['silhouette'].max():.4f})")
    print(f"  Plus grande dist_c → {df_ok['dist_c'].idxmax()}  ({df_ok['dist_c'].max():.3f})")

df_res.to_csv(os.path.join(OUTPUT_DIR, 'batch_pipeline_results.csv'))
print('\nTableau sauvegardé → batch_pipeline_results.csv')


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# MOYENNE DES SCORES — résumé synthétique
# ══════════════════════════════════════════════════════════════════════
df_ok = df_res.dropna(subset=['knn_acc_%'])

metrics = {
    'KNN acc (%)'        : 'knn_acc_%',
    'F1 macro (%)'       : 'f1_%',
    'Silhouette 2D'      : 'silhouette',
    'Distance centroïdes': 'dist_c',
    'Loss 2D finale'     : 'loss_2d',
    'Loss 32D finale'    : 'loss_32d',
}

print('═' * 45)
print('  MOYENNES SUR LES FICHIERS VALIDES')
print(f'  ({len(df_ok)}/{len(df_res)} fichiers pris en compte)')
print('═' * 45)
for label, col in metrics.items():
    vals = df_ok[col].dropna()
    if len(vals):
        print(f'  {label:<25} {vals.mean():>8.3f}  ±{vals.std():>7.3f}')
    else:
        print(f'  {label:<25} {"—":>8}')
print('═' * 45)